**Data loading and Preprocessing**

In [2]:
import pandas as pd

file = "CU235-XLS-ENG_student spreadsheet-1.xlsx"
xls = pd.ExcelFile(file)

print(xls.sheet_names)


['Columbia CaseWorks', 'Data for Analysis', 'Base Price', 'Product Component Matrix', 'Actual demand', '1 mo forecast', '2 mo forecast', '3 mo forecast', '4 mo forecast', '5 mo forecast', '6 mo forecast']


In [3]:
actual_raw = pd.read_excel(file, sheet_name="Actual demand", header=None)
fc2_raw = pd.read_excel(file, sheet_name="2 mo forecast", header=None)
fc6_raw = pd.read_excel(file, sheet_name="6 mo forecast", header=None)

actual_raw.head(10)


,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,QUARTERLY DEMAND DATA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Segment,Product,4Q 12,1Q 13,2Q 13,3Q 13,4Q 13,1Q 14,2Q 14,3Q 14,...,4Q 15,1Q 16,2Q 16,3Q 16,4Q 16,1Q 17,2Q 17,NaN,Mean,Std Dev
3,PREMIUM,750w,23,31,20,32,41,0,0,0,...,12,22,30,28,27,26,20,NaN,16.789474,13.70555
4,NaN,HDxt,9,6,6,9,9,3,9,4,...,0,3,1,2,0,3,1,NaN,3.894737,3.264195
5,NaN,450w,71,59,56,54,78,63,64,67,...,76,59,69,68,72,38,50,NaN,62.263158,10.913
6,NaN,HDi,13,14,14,22,0,4,11,15,...,3,3,8,6,7,4,1,NaN,8.052632,5.806832
7,PERFORMANCE,Pioneer,0,0,0,0,0,0,0,0,...,10,12,17,14,14,14,14,NaN,5.315789,6.733438
8,NaN,Voyager,3,7,9,10,6,8,16,15,...,13,14,2,3,7,12,23,NaN,10.210526,5.662537
9,VALUE,355 BRIVO/360 Optima,44,35,44,49,44,39,41,55,...,71,32,37,30,35,31,21,NaN,43.578947,12.902695


In [4]:
def find_header_row(df):
    for i in range(15):
        row_values = df.iloc[i].astype(str).str.strip().tolist()
        if "Segment" in row_values and "Product" in row_values:
            return i
    return None

print(find_header_row(actual_raw))
print(find_header_row(fc2_raw))
print(find_header_row(fc6_raw))



2
2
2


In [5]:
actual_header = find_header_row(actual_raw)
fc2_header = find_header_row(fc2_raw)
fc6_header = find_header_row(fc6_raw)

actual = pd.read_excel(file, sheet_name="Actual demand", header=actual_header)
fc2 = pd.read_excel(file, sheet_name="2 mo forecast", header=fc2_header)
fc6 = pd.read_excel(file, sheet_name="6 mo forecast", header=fc6_header)

print(actual.head())
print(fc2.head())
print(fc6.head())


       Segment  Product  4Q 12  1Q 13  2Q 13  3Q 13  4Q 13  1Q 14  2Q 14  \
0      PREMIUM     750w   23.0   31.0   20.0   32.0   41.0    0.0    0.0   
1          NaN     HDxt    9.0    6.0    6.0    9.0    9.0    3.0    9.0   
2          NaN     450w   71.0   59.0   56.0   54.0   78.0   63.0   64.0   
3          NaN      HDi   13.0   14.0   14.0   22.0    0.0    4.0   11.0   
4  PERFORMANCE  Pioneer    0.0    0.0    0.0    0.0    0.0    0.0    0.0   

   3Q 14  ...  4Q 15  1Q 16  2Q 16  3Q 16 4Q 16  1Q 17  2Q 17  Unnamed: 21  \
0    0.0  ...     12   22.0   30.0   28.0  27.0   26.0   20.0          NaN   
1    4.0  ...      0    3.0    1.0    2.0   0.0    3.0    1.0          NaN   
2   67.0  ...     76   59.0   69.0   68.0  72.0   38.0   50.0          NaN   
3   15.0  ...      3    3.0    8.0    6.0   7.0    4.0    1.0          NaN   
4    0.0  ...     10   12.0   17.0   14.0  14.0   14.0   14.0          NaN   

        Mean    Std Dev  
0  16.789474  13.705550  
1   3.894737   3.26419

In [6]:
quarter_cols = [
    "4Q 12","1Q 13","2Q 13","3Q 13","4Q 13",
    "1Q 14","2Q 14","3Q 14","4Q 14",
    "1Q 15","2Q 15","3Q 15","4Q 15",
    "1Q 16","2Q 16","3Q 16","4Q 16",
    "1Q 17","2Q 17"
]

def clean_sheet(df):
    df = df.copy()
    df["Segment"] = df["Segment"].ffill()
    df["Product"] = df["Product"].astype(str).str.strip()
    df = df[["Segment", "Product"] + quarter_cols]
    df = df.dropna(subset=["Product"])
    return df

actual = clean_sheet(actual)
fc2 = clean_sheet(fc2)
fc6 = clean_sheet(fc6)

print(actual.head())


       Segment  Product  4Q 12  1Q 13  2Q 13  3Q 13  4Q 13  1Q 14  2Q 14  \
0      PREMIUM     750w   23.0   31.0   20.0   32.0   41.0    0.0    0.0   
1      PREMIUM     HDxt    9.0    6.0    6.0    9.0    9.0    3.0    9.0   
2      PREMIUM     450w   71.0   59.0   56.0   54.0   78.0   63.0   64.0   
3      PREMIUM      HDi   13.0   14.0   14.0   22.0    0.0    4.0   11.0   
4  PERFORMANCE  Pioneer    0.0    0.0    0.0    0.0    0.0    0.0    0.0   

   3Q 14  ...  1Q 15  2Q 15  3Q 15  4Q 15 1Q 16  2Q 16  3Q 16  4Q 16  1Q 17  \
0    0.0  ...    0.0    4.0    3.0     12  22.0   30.0   28.0   27.0   26.0   
1    4.0  ...    3.0    1.0    0.0      0   3.0    1.0    2.0    0.0    3.0   
2   67.0  ...   43.0   55.0   69.0     76  59.0   69.0   68.0   72.0   38.0   
3   15.0  ...    8.0    2.0    6.0      3   3.0    8.0    6.0    7.0    4.0   
4    0.0  ...    0.0    0.0    6.0     10  12.0   17.0   14.0   14.0   14.0   

   2Q 17  
0   20.0  
1    1.0  
2   50.0  
3    1.0  
4   14.0  

[

In [7]:
def to_long(df, value_name):
    return df.melt(
        id_vars=["Segment", "Product"],
        value_vars=quarter_cols,
        var_name="Quarter",
        value_name=value_name
    )

actual_long = to_long(actual, "Actual")
fc2_long = to_long(fc2, "Forecast_2m")
fc6_long = to_long(fc6, "Forecast_6m")

print(actual_long.head())
print(fc2_long.head())
print(fc6_long.head())


       Segment  Product Quarter Actual
0      PREMIUM     750w   4Q 12   23.0
1      PREMIUM     HDxt   4Q 12    9.0
2      PREMIUM     450w   4Q 12   71.0
3      PREMIUM      HDi   4Q 12   13.0
4  PERFORMANCE  Pioneer   4Q 12    0.0
       Segment  Product Quarter  Forecast_2m
0      PREMIUM     750w   4Q 12         26.0
1      PREMIUM     HDxt   4Q 12          8.0
2      PREMIUM     450w   4Q 12         67.0
3      PREMIUM      HDi   4Q 12         12.0
4  PERFORMANCE  Pioneer   4Q 12          0.0
       Segment  Product Quarter  Forecast_6m
0      PREMIUM     750w   4Q 12         30.0
1      PREMIUM     HDxt   4Q 12         12.0
2      PREMIUM     450w   4Q 12         65.0
3      PREMIUM      HDi   4Q 12         14.0
4  PERFORMANCE  Pioneer   4Q 12          0.0


In [8]:
df = actual_long.merge(
    fc2_long,
    on=["Segment", "Product", "Quarter"],
    how="inner"
).merge(
    fc6_long,
    on=["Segment", "Product", "Quarter"],
    how="inner"
)

print(df.head())
print(df.shape)


       Segment  Product Quarter Actual  Forecast_2m  Forecast_6m
0      PREMIUM     750w   4Q 12   23.0         26.0         30.0
1      PREMIUM     HDxt   4Q 12    9.0          8.0         12.0
2      PREMIUM     450w   4Q 12   71.0         67.0         65.0
3      PREMIUM      HDi   4Q 12   13.0         12.0         14.0
4  PERFORMANCE  Pioneer   4Q 12    0.0          0.0          0.0
(171, 6)


In [9]:
df["Segment"] = df["Segment"].astype(str).str.strip()
df["Product"] = df["Product"].astype(str).str.strip()

for col in ["Actual", "Forecast_2m", "Forecast_6m"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)
print(df.isna().sum())


Segment         object
Product         object
Quarter         object
Actual         float64
Forecast_2m    float64
Forecast_6m    float64
dtype: object
Segment        0
Product        0
Quarter        0
Actual         0
Forecast_2m    0
Forecast_6m    0
dtype: int64


In [10]:
df.to_csv("q2_python_starter.csv", index=False)


**Task 2**

In [11]:
import numpy as np

for horizon in ["2m", "6m"]:
    fcol = f"Forecast_{horizon}"
    df[f"Error_{horizon}"] = df[fcol] - df["Actual"]
    df[f"AbsError_{horizon}"] = df[f"Error_{horizon}"].abs()
    df[f"SqError_{horizon}"] = df[f"Error_{horizon}"] ** 2
    df[f"APE_{horizon}"] = np.where(
        df["Actual"] != 0,
        df[f"AbsError_{horizon}"] / df["Actual"],
        np.nan
    )

df.head()


,Segment,Product,Quarter,Actual,Forecast_2m,Forecast_6m,Error_2m,AbsError_2m,SqError_2m,APE_2m,Error_6m,AbsError_6m,SqError_6m,APE_6m
0,PREMIUM,750w,4Q 12,23.0,26.0,30.0,3.0,3.0,9.0,0.130435,7.0,7.0,49.0,0.304348
1,PREMIUM,HDxt,4Q 12,9.0,8.0,12.0,-1.0,1.0,1.0,0.111111,3.0,3.0,9.0,0.333333
2,PREMIUM,450w,4Q 12,71.0,67.0,65.0,-4.0,4.0,16.0,0.056338,-6.0,6.0,36.0,0.084507
3,PREMIUM,HDi,4Q 12,13.0,12.0,14.0,-1.0,1.0,1.0,0.076923,1.0,1.0,1.0,0.076923
4,PERFORMANCE,Pioneer,4Q 12,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,NaN


In [12]:
def accuracy_metrics(actual, forecast):
    error = forecast - actual
    abs_error = np.abs(error)
    sq_error = error ** 2
    ape = np.where(actual != 0, abs_error / actual, np.nan)

    return {
        "ME": error.mean(),
        "MAE": abs_error.mean(),
        "MSE": sq_error.mean(),
        "RMSE": np.sqrt(sq_error.mean()),
        "WAPE": abs_error.sum() / actual.sum(),
        "MAPE": np.nanmean(ape)
    }


In [16]:
overall = pd.DataFrame([
    {"Horizon": "2 months", **accuracy_metrics(df["Actual"], df["Forecast_2m"])},
    {"Horizon": "6 months", **accuracy_metrics(df["Actual"], df["Forecast_6m"])}
])

print(overall.round(4))


    Horizon      ME     MAE      MSE    RMSE    WAPE    MAPE
0  2 months  2.5533  3.4082  43.3013  6.5804  0.1757  0.2502
1  6 months  2.9766  4.9766  68.3801  8.2692  0.2566  0.5168


In [13]:
by_product = []

for product, sub in df.groupby("Product"):
    by_product.append({
        "Product": product,
        "Horizon": "2 months",
        **accuracy_metrics(sub["Actual"], sub["Forecast_2m"])
    })
    by_product.append({
        "Product": product,
        "Horizon": "6 months",
        **accuracy_metrics(sub["Actual"], sub["Forecast_6m"])
    })

by_product = pd.DataFrame(by_product)
print(by_product.round(4))


                 Product   Horizon      ME     MAE       MSE     RMSE    WAPE  \
0   355 BRIVO/360 Optima  2 months  1.2105  2.3684   10.8947   3.3007  0.0543   
1   355 BRIVO/360 Optima  6 months  6.1579  9.3158  133.4211  11.5508  0.2138   
2                   450w  2 months  4.6333  6.7386   59.9301   7.7415  0.1082   
3                   450w  6 months  5.4737  8.7368  100.7368  10.0368  0.1403   
4                   750w  2 months  1.8369  2.2580   12.0115   3.4658  0.1345   
5                   750w  6 months  1.7368  3.8421   27.4211   5.2365  0.2288   
6      Creator - 8 Chnl.  2 months  5.7407  5.7407  112.7387  10.6178  0.5003   
7      Creator - 8 Chnl.  6 months  5.5789  7.0526  180.8421  13.4478  0.6147   
8    Explorer - 16 Chnl.  2 months  6.7739  7.0896  170.8207  13.0698  0.5454   
9    Explorer - 16 Chnl.  6 months  1.7895  4.8421  111.4737  10.5581  0.3725   
10                   HDi  2 months  0.2632  1.5263    3.1053   1.7622  0.1895   
11                   HDi  6 

In [14]:
by_segment = []

for segment, sub in df.groupby("Segment"):
    by_segment.append({
        "Segment": segment,
        "Horizon": "2 months",
        **accuracy_metrics(sub["Actual"], sub["Forecast_2m"])
    })
    by_segment.append({
        "Segment": segment,
        "Horizon": "6 months",
        **accuracy_metrics(sub["Actual"], sub["Forecast_6m"])
    })

by_segment = pd.DataFrame(by_segment)
print(by_segment.round(4))


       Segment   Horizon      ME     MAE       MSE     RMSE    WAPE    MAPE
0  PERFORMANCE  2 months  1.3395  2.0289    9.2897   3.0479  0.2614  0.2777
1  PERFORMANCE  6 months  2.2105  2.4737   14.3158   3.7836  0.3186  0.4394
2      PREMIUM  2 months  1.6439  2.8544   19.1696   4.3783  0.1255  0.2145
3      PREMIUM  6 months  2.2105  4.6579   40.2632   6.3453  0.2047  0.6194
4        VALUE  2 months  4.5750  5.0663   98.1514   9.9071  0.2233  0.2948
5        VALUE  6 months  4.5088  7.0702  141.9123  11.9127  0.3117  0.3874


In [15]:
def bias_counts(actual, forecast):
    error = forecast - actual
    return {
        "Underforecast_rate": (error < 0).mean(),
        "Overforecast_rate": (error > 0).mean(),
        "Exact_rate": (error == 0).mean()
    }

bias_summary = pd.DataFrame([
    {"Horizon": "2 months", **bias_counts(df["Actual"], df["Forecast_2m"])},
    {"Horizon": "6 months", **bias_counts(df["Actual"], df["Forecast_6m"])}
])

print(bias_summary.round(4))


    Horizon  Underforecast_rate  Overforecast_rate  Exact_rate
0  2 months              0.2164             0.5205      0.2632
1  6 months              0.2222             0.5146      0.2632


**Optional export of table to csv**

In [ ]:
#overall.to_csv("q2_overall_metrics.csv", index=False)
#by_product.to_csv("q2_by_product_metrics.csv", index=False)
#by_segment.to_csv("q2_by_segment_metrics.csv", index=False)
#bias_summary.to_csv("q2_bias_summary.csv", index=False)


**Task 3**

In [ ]:
# Before = 6-month commitment
df["Excess_6m"] = np.maximum(df["Forecast_6m"] - df["Actual"], 0)
df["Shortage_6m"] = np.maximum(df["Actual"] - df["Forecast_6m"], 0)

# After = 2-month commitment
df["Excess_2m"] = np.maximum(df["Forecast_2m"] - df["Actual"], 0)
df["Shortage_2m"] = np.maximum(df["Actual"] - df["Forecast_2m"], 0)

task3_summary = pd.DataFrame({
    "Scenario": ["Before: 6m commit", "After: 2m commit"],
    "Total_excess": [df["Excess_6m"].sum(), df["Excess_2m"].sum()],
    "Total_shortage": [df["Shortage_6m"].sum(), df["Shortage_2m"].sum()],
    "Total_mismatch": [
        df["Excess_6m"].sum() + df["Shortage_6m"].sum(),
        df["Excess_2m"].sum() + df["Shortage_2m"].sum()
    ]
})

#Service level proxy: proportion of demand that is fulfilled by the forecast ( min of forecast and actual, summed across all rows, divided by total demand)
fulfilled_6m = np.minimum(df["Forecast_6m"], df["Actual"]).sum()
fulfilled_2m = np.minimum(df["Forecast_2m"], df["Actual"]).sum()
total_demand = df["Actual"].sum()

task3_summary["Service_proxy"] = [
    fulfilled_6m / total_demand,
    fulfilled_2m / total_demand
]
#Mismatch cost index: a weighted sum of excess and shortage, where shortage is more expensive than excess.
shortage_penalty = 1.0
excess_penalty = 0.35

task3_summary["Mismatch_cost_index"] = [
    shortage_penalty * df["Shortage_6m"].sum() + excess_penalty * df["Excess_6m"].sum(),
    shortage_penalty * df["Shortage_2m"].sum() + excess_penalty * df["Excess_2m"].sum()
]
task3_summary


,Scenario,Total_excess,Total_shortage,Total_mismatch,Service_proxy,Mismatch_cost_index
0,Before: 6m commit,680.000000,171.000000,851.000000,0.948447,409.00000
1,After: 2m commit,509.711454,73.098901,582.810355,0.977962,251.49791
